# Exploratory Data Analysis and Feature-Group Inspection for HIGGS Classification

**Project:** Higgs Event Lab | **Module:** MID-02

This notebook performs reproducible EDA on the documented 250,000-row analysis sample by default, using the original UCI HIGGS source file. Set `HIGGS_ANALYSIS_ROWS=11000000` before setup for full-row EDA if sufficient RAM is available.

> **Scientific boundary:** the rows are Monte Carlo simulations. This is exploratory classification analysis, not a discovery claim.

## Step 0 — Project setup and data acquisition

The setup cells locate/clone the repository and validate the original UCI source file. EDA uses a configurable 250,000-row analysis sample by default to keep memory usage manageable; set `HIGGS_ANALYSIS_ROWS=11000000` before running setup only if full-data EDA is intended.

In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path
from importlib.metadata import PackageNotFoundError, version
from packaging.requirements import Requirement
from packaging.version import Version

REPO_URL = "https://github.com/CosmicMagnetar/higgs-event-lab.git"
cwd = Path.cwd().resolve()


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:
candidates = []
if os.environ.get("HIGGS_REPO_ROOT"):
    candidates.append(Path(os.environ["HIGGS_REPO_ROOT"]).expanduser().resolve())
candidates.extend([cwd, cwd / "higgs-event-lab", *cwd.parents])

def is_higgs_repo(path):
    return (path / "src" / "higgs_event_lab").is_dir() and (path / "pyproject.toml").is_file()


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:
root = next((path for path in dict.fromkeys(candidates) if is_higgs_repo(path)), None)
if root is None:
    repo_target = cwd / "higgs-event-lab"
    suffix = 1
    while repo_target.exists() and not is_higgs_repo(repo_target):
        if repo_target.is_dir() and not any(repo_target.iterdir()):
            break
        suffix += 1
        repo_target = cwd / f"higgs-event-lab-{suffix}"
    print("Cloning Higgs Event Lab into:", repo_target)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo_target)], check=True)
    root = repo_target.resolve()

if not is_higgs_repo(root):
    raise FileNotFoundError(f"Could not locate a valid Higgs Event Lab checkout at {root}.")


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:
src_dir = root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
os.environ["HIGGS_REPO_ROOT"] = str(root)
os.chdir(root)
print("Project root:", root)


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:
REQUIRED_PACKAGES = [
    ("requests", "requests>=2.31"),
    ("numpy", "numpy>=1.26"),
    ("pandas", "pandas>=2.1"),
    ("sklearn", "scikit-learn>=1.4"),
    ("matplotlib", "matplotlib>=3.8"),
]
missing_or_old = []
for module_name, requirement_text in REQUIRED_PACKAGES:
    requirement = Requirement(requirement_text)
    try:
        installed = Version(version(requirement.name))
    except PackageNotFoundError:
        installed = None
    if installed is None or (requirement.specifier and installed not in requirement.specifier):
        missing_or_old.append(requirement_text)
if missing_or_old:
    print("Installing required packages:", ", ".join(missing_or_old))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing_or_old], check=True)
else:
    print("Required packages already satisfy the minimum versions.")


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:
raw_path = root / "data" / "raw" / "HIGGS.csv.gz"
zip_path = raw_path.parent / "higgs_download.zip"
temp_raw_path = raw_path.with_name(raw_path.name + ".part")
validation_marker = raw_path.with_name(raw_path.name + ".validation.json")
raw_path.parent.mkdir(parents=True, exist_ok=True)
output_dir = root / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)
url = "https://archive.ics.uci.edu/static/public/280/higgs.zip"
full_row_count = 11_000_000
analysis_row_count = int(os.environ.get("HIGGS_ANALYSIS_ROWS", "250000"))
if not 10 <= analysis_row_count <= full_row_count:
    raise ValueError(f"HIGGS_ANALYSIS_ROWS must be between 10 and {full_row_count:,}.")
processed_path = raw_path
print("Raw dataset path:", raw_path)
print(f"Analysis rows: {analysis_row_count:,}")


## Step 0B — Resumable download helpers

This cell only defines the ZIP/GZIP validation and retry/resume functions. It does not start a download.

This section records the purpose, expected output and interpretation of the computation so the result can be checked independently.


In [ ]:
import re
import time
import zipfile
from pathlib import Path


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:

def find_csv_gz_member(archive_path):
    """Return the CSV.GZ member only when the ZIP directory is readable."""
    archive_path = Path(archive_path)
    if not archive_path.is_file():
        return None
    try:
        with zipfile.ZipFile(archive_path, "r") as archive:
            return next((name for name in archive.namelist() if name.lower().endswith(".csv.gz")), None)
    except (OSError, zipfile.BadZipFile, EOFError):
        return None


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:

def _download_mode(response, existing, destination):
    if response.status_code == 416:
        Path(destination).unlink(missing_ok=True)
        raise IOError("The server rejected the resume offset; restarting from zero.")
    response.raise_for_status()
    append = existing > 0 and response.status_code == 206
    if append:
        content_range = response.headers.get("Content-Range", "")
        match = re.match(r"bytes\s+(\d+)-", content_range)
        if not match or int(match.group(1)) != existing:
            Path(destination).unlink(missing_ok=True)
            raise IOError("Server returned an unexpected byte range; restarting from zero.")
    return "ab" if append else "wb"


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:

def _download_attempt(download_url, destination, session):
    destination = Path(destination)
    existing = destination.stat().st_size if destination.exists() else 0
    headers = {"Range": f"bytes={existing}-"} if existing else {}
    with session.get(download_url, headers=headers, stream=True, timeout=(30, 180)) as response:
        mode = _download_mode(response, existing, destination)
        expected = response.headers.get("Content-Length")
        received = 0
        with destination.open(mode) as target:
            for chunk in response.iter_content(chunk_size=4 * 1024 * 1024):
                if chunk:
                    target.write(chunk)
                    received += len(chunk)
        if expected is not None and received != int(expected):
            raise IOError(f"Incomplete HTTP response: received {received:,} of {int(expected):,} bytes.")
    if not find_csv_gz_member(destination):
        raise IOError("The downloaded ZIP is not complete yet.")


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:

def download_zip_resumable(download_url, destination, max_attempts=10):
    """Download with retries and HTTP Range resume where the server supports it."""
    import requests
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if find_csv_gz_member(destination):
        print("A readable ZIP is already present; reusing it.")
        return destination
    session = requests.Session()
    last_error = None
    for attempt in range(1, max_attempts + 1):
        try:
            _download_attempt(download_url, destination, session)
            print(f"ZIP received: {destination.stat().st_size / (1024**3):.2f} GiB.")
            return destination
        except Exception as exc:
            last_error = exc
            print(f"Download attempt {attempt}/{max_attempts} failed: {exc}")
            if attempt < max_attempts:
                time.sleep(min(2 ** (attempt - 1), 30))
    raise RuntimeError(f"Could not obtain a readable UCI ZIP. Rerun this cell. Last error: {last_error}")


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:

def count_gzip_rows(path, chunk_size=4 * 1024 * 1024):
    """Fully read a gzip file and count CSV rows without loading it into memory."""
    import gzip
    newline_count = 0
    last_byte = b""
    with gzip.open(path, "rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            newline_count += chunk.count(b"\n")
            last_byte = chunk[-1:]
    if last_byte and last_byte != b"\n":
        newline_count += 1
    return newline_count


## Step 0C — Download, extract, and validate the raw HIGGS file

This cell reuses a previously validated file, repairs old partial files, or downloads and validates the complete 11-million-row source. Rerun this cell after a network failure.

In [ ]:
import json
import shutil
import zipfile


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:

def _write_validation_marker(path, rows):
    marker = Path(path).with_name(Path(path).name + ".validation.json")
    marker.write_text(
        json.dumps({"rows": int(rows), "size_bytes": int(Path(path).stat().st_size)}, indent=2),
        encoding="utf-8",
    )


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:

def _existing_raw_is_valid(path, expected_rows, marker_path):
    path = Path(path)
    if not path.is_file() or path.stat().st_size == 0:
        return False
    try:
        metadata = json.loads(Path(marker_path).read_text(encoding="utf-8"))
        if metadata.get("rows") == expected_rows and metadata.get("size_bytes") == path.stat().st_size:
            with path.open("rb") as handle:
                return handle.read(2) == b"\x1f\x8b"
    except (OSError, ValueError, TypeError):
        pass
    try:
        rows = count_gzip_rows(path)
        if rows != expected_rows:
            print(f"Existing raw file is incomplete: expected {expected_rows:,}, found {rows:,} rows.")
            return False
        _write_validation_marker(path, rows)
        return True
    except Exception as exc:
        print("Existing raw file failed gzip validation:", exc)
        return False


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:

def _extract_zip_member(zip_path, member_name, temp_path):
    temp_path.unlink(missing_ok=True)
    with zipfile.ZipFile(zip_path, "r") as archive:
        with archive.open(member_name, "r") as source, temp_path.open("wb") as target:
            shutil.copyfileobj(source, target, length=4 * 1024 * 1024)
    return count_gzip_rows(temp_path)


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:

def _extract_and_validate_raw(url, zip_path, raw_path, temp_raw_path, marker_path, expected_rows):
    gz_name = find_csv_gz_member(zip_path)
    if not gz_name:
        print("No complete ZIP found; downloading or resuming the UCI archive.")
        download_zip_resumable(url, zip_path, max_attempts=10)
        gz_name = find_csv_gz_member(zip_path)
    if not gz_name:
        raise RuntimeError("No readable .csv.gz member found. Rerun the download helper cell.")
    print("Extracting dataset member:", gz_name)
    try:
        rows = _extract_zip_member(zip_path, gz_name, temp_raw_path)
        if rows != expected_rows:
            raise ValueError(f"Expected {expected_rows:,} rows, found {rows:,}.")
        temp_raw_path.replace(raw_path)
        _write_validation_marker(raw_path, rows)
    except Exception as exc:
        temp_raw_path.unlink(missing_ok=True)
        zip_path.unlink(missing_ok=True)
        raise RuntimeError("Extraction/validation failed. Incomplete files were removed; rerun this cell.") from exc
    zip_path.unlink(missing_ok=True)
    return rows


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:
complete_row_count = None
if raw_path.exists() and _existing_raw_is_valid(raw_path, full_row_count, validation_marker):
    complete_row_count = full_row_count
    print("A previously validated complete HIGGS file is present; skipping download.")
else:
    raw_path.unlink(missing_ok=True)
    validation_marker.unlink(missing_ok=True)
    complete_row_count = _extract_and_validate_raw(
        url, zip_path, raw_path, temp_raw_path, validation_marker, full_row_count
    )
if complete_row_count != full_row_count:
    raise ValueError(f"Dataset has {complete_row_count:,} rows; expected {full_row_count:,}.")
print("Raw HIGGS data ready:", raw_path)
print(f"Validated row count: {complete_row_count:,}")
print(f"Compressed size: {raw_path.stat().st_size / (1024**3):.2f} GiB")


## Step 1 — Load the configured analysis sample

This section records the purpose, expected output and interpretation of the computation so the result can be checked independently.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

from higgs_event_lab.data import load_higgs_sample
from higgs_event_lab.features import (
    ALL_COLUMNS,
    ALL_FEATURES,
    FEATURE_GROUPS,
    HIGH_LEVEL_FEATURES,
    LOW_LEVEL_FEATURES,
    TARGET_COLUMN,
    get_feature_columns,
)

data_path = raw_path
batch = load_higgs_sample(data_path, n_rows=analysis_row_count, feature_group="all")
X = batch.X
y = batch.y

print("Raw source rows available:", f"{full_row_count:,}")


## Execution step

This code cell performs one reproducible part of the workflow. It uses the shared project modules, preserves the documented protocol, and prints inspectable output for the next analysis step.


In [ ]:
print("Analysis sample rows:", f"{analysis_row_count:,}")
print("Feature matrix shape:", X.shape)
print("Target label vector shape:", y.shape)
print("Rows loaded for this analysis sample:", f"{len(X):,}")
print("Low-level feature count:", len(LOW_LEVEL_FEATURES))
print("High-level feature count:", len(HIGH_LEVEL_FEATURES))


## 2. Dataset Schema and Target Label Semantics

**Question:** What do the binary target labels represent, and how are the input columns partitioned across physical subsystems?

**Why it matters:** Precise semantic clarity prevents confusion between the 2012 Higgs boson discovery and this exotic-Higgs benchmark. In this benchmark,  denotes simulated signal events with hypothetical heavy particles, while  denotes standard model top-pair background events with identical decay products.

**Explanation:** We inspect the exact column breakdown and verify that the target column  is never present inside the feature DataFrame.

**Expected output:** Feature group breakdown confirming 21 low-level kinematic variables and 7 high-level invariant masses.

In [ ]:
assert TARGET_COLUMN not in X.columns
print("Target column name:", TARGET_COLUMN)
print("Unique label values:", np.unique(y))

group_summary = pd.DataFrame([
    {"Group": "Target", "Count": 1, "Variables": TARGET_COLUMN},
    {"Group": "Low-level kinematics", "Count": len(LOW_LEVEL_FEATURES), "Variables": f"{LOW_LEVEL_FEATURES[0]} ... {LOW_LEVEL_FEATURES[-1]}"},
    {"Group": "High-level invariant masses", "Count": len(HIGH_LEVEL_FEATURES), "Variables": f"{HIGH_LEVEL_FEATURES[0]} ... {HIGH_LEVEL_FEATURES[-1]}"},
])
print(group_summary.to_string(index=False))


## 3. Data Integrity and Missing Value Verification

**Question:** Are there any null, missing, NaN, or non-finite values present in the loaded HIGGS sample?

**Why it matters:** Missing values in collider reconstruction can indicate corrupted sensor streams or failed kinematic fits. Downstream algorithms like Logistic Regression and compact MLPs will produce invalid gradients or throw exceptions if non-finite entries exist.

**Explanation:** We run exhaustive checks across all 28 feature columns and the label array for nulls, infinite values, or improper data types.

**Expected output:** Zero missing values, zero infinite values, and uniform float64 numerical representations.

In [ ]:
null_counts = X.isnull().sum()
total_nulls = int(null_counts.sum())
infinite_counts = int(np.isinf(X.to_numpy()).sum())
label_nulls = int(pd.Series(y).isnull().sum())

print("Total missing values across all features:", total_nulls)
print("Total infinite values across all features:", infinite_counts)
print("Total missing values in target array:", label_nulls)
print("All columns numeric dtype:", all(np.issubdtype(dt, np.number) for dt in X.dtypes))


## 4. Class Balance and Signal Fraction Analysis

**Question:** What is the exact class distribution between signal and background events in this empirical sample?

**Why it matters:** Many machine learning benchmarks suffer from extreme class imbalance requiring synthetic resampling or asymmetric cost weighting. Previous project documentation noted that class balance was unverified (BUG-014); we compute the exact empirical signal fraction to ground the project facts.

**Explanation:** We compute the counts, percentages, and imbalance ratio between signal (class 1) and background (class 0) collision events.

**Expected output:** Empirical count and percentage for both classes confirming approximate balance.

In [ ]:
label_counts = pd.Series(y).value_counts().sort_index()
signal_count = int(label_counts.get(1, 0))
bkg_count = int(label_counts.get(0, 0))
total_count = len(y)
signal_fraction = signal_count / total_count

print(f"Background (class 0): {bkg_count:,} ({bkg_count / total_count:.2%})")
print(f"Signal (class 1):     {signal_count:,} ({signal_count / total_count:.2%})")
print(f"Empirical signal fraction: {signal_fraction:.4f}")
print(f"Imbalance ratio (Signal / Background): {signal_count / bkg_count:.4f}")


## 5. Discrete Feature Verification: Jet B-tagging Properties

**Question:** Do the jet b-tagging variables exhibit continuous or discrete values, and what physical meaning does this convey?

**Why it matters:** Physics documentation (BUG-014 and section 4 of the dataset guide) indicated that b-tagging might be discrete rather than continuous. Machine learning preprocessing must treat discrete features appropriately rather than assuming smooth Gaussian variations.

**Explanation:** We examine unique value frequencies and zero-fraction metrics across the four b-tagging columns (, , , ).

**Expected output:** Table showing unique value counts and the predominance of discrete values.

In [ ]:
btag_cols = ["jet_1_btag", "jet_2_btag", "jet_3_btag", "jet_4_btag"]
btag_stats = []
for col in btag_cols:
    vals = X[col].value_counts(normalize=True).head(5)
    zero_pct = float((X[col] == 0.0).mean())
    btag_stats.append({
        "Feature": col,
        "Unique count": X[col].nunique(),
        "Zero fraction": f"{zero_pct:.2%}",
        "Top value": f"{vals.index[0]:.2f} ({vals.iloc[0]:.1%})",
    })
print(pd.DataFrame(btag_stats).to_string(index=False))


## 6. Kinematic Feature Distributions: Signal versus Background

**Question:** How do low-level particle kinematic variables (transverse momenta, pseudorapidity, azimuthal angles) differ between signal and background?

**Why it matters:** Understanding the physical separation in 1D projections highlights why linear models struggle on raw kinematics alone. For instance, azimuthal angles are uniformly distributed due to detector cylindrical symmetry, whereas transverse momenta exhibit heavy tails.

**Explanation:** We compute summary statistics (mean and standard deviation) split by signal and background across representative low-level features.

**Expected output:** Comparative tabular statistics showing subtle shifts in transverse momenta and near identical angular spreads.

In [ ]:
kinematic_subset = ["lepton_pt", "lepton_eta", "missing_energy_magnitude", "jet_1_pt"]
kin_comparison = []
for col in kinematic_subset:
    sig_mean = float(X.loc[y == 1, col].mean())
    sig_std = float(X.loc[y == 1, col].std())
    bkg_mean = float(X.loc[y == 0, col].mean())
    bkg_std = float(X.loc[y == 0, col].std())
    kin_comparison.append({
        "Feature": col,
        "Signal Mean ± Std": f"{sig_mean:.3f} ± {sig_std:.3f}",
        "Bkg Mean ± Std": f"{bkg_mean:.3f} ± {bkg_std:.3f}",
        "Abs Diff": f"{abs(sig_mean - bkg_mean):.3f}",
    })
print(pd.DataFrame(kin_comparison).to_string(index=False))


## 7. High-Level Derived Invariant Mass Distributions

**Question:** Do physicist-engineered invariant masses provide clearer discrimination between exotic Higgs decay signals and background top-quark processes?

**Why it matters:** Invariant masses represent composite particle resonance signatures. For example,  reconstructs candidate Higgs boson candidates decaying into bottom quark pairs, while  reconstructs top decay chains.

**Explanation:** We compare signal and background distributions for key invariant masses (, , , ), calculating separation metrics between the two class distributions.

**Expected output:** Statistical comparisons demonstrating noticeable distributional shifts in invariant masses for signal events compared to background.

In [ ]:
mass_cols = list(HIGH_LEVEL_FEATURES)[:4]
mass_comparison = []
for col in mass_cols:
    sig_mean = float(X.loc[y == 1, col].mean())
    sig_std = float(X.loc[y == 1, col].std())
    bkg_mean = float(X.loc[y == 0, col].mean())
    bkg_std = float(X.loc[y == 0, col].std())
    diff = abs(sig_mean - bkg_mean)
    mass_comparison.append({
        "High-level Feature": col,
        "Signal Mean ± Std": f"{sig_mean:.3f} ± {sig_std:.3f}",
        "Bkg Mean ± Std": f"{bkg_mean:.3f} ± {bkg_std:.3f}",
        "Separation": f"{diff / ((sig_std + bkg_std) / 2):.3f}",
    })
print(pd.DataFrame(mass_comparison).to_string(index=False))


## 8. Correlation Structure Across Feature Groups

**Question:** How strongly correlated are low-level kinematic variables compared to high-level derived invariant mass features?

**Why it matters:** High collinearity among features impacts linear model stability and coefficient interpretability. Tree models and regularized neural networks handle collinearity differently, making it crucial to map the cross-feature correlation structure.

**Explanation:** We compute the Pearson correlation matrix for the high-level features and analyze the strongest correlations between high-level and low-level variables.

**Expected output:** Summary of highest correlated feature pairs.

In [ ]:
corr_matrix = X.corr(method="pearson")
high_level_corr = corr_matrix.loc[list(HIGH_LEVEL_FEATURES), list(HIGH_LEVEL_FEATURES)]

pairs = []
for i in range(len(HIGH_LEVEL_FEATURES)):
    for j in range(i + 1, len(HIGH_LEVEL_FEATURES)):
        f1, f2 = HIGH_LEVEL_FEATURES[i], HIGH_LEVEL_FEATURES[j]
        val = high_level_corr.loc[f1, f2]
        pairs.append((f1, f2, abs(val), val))

pairs.sort(key=lambda item: item[2], reverse=True)
top_corr_df = pd.DataFrame([
    {"Feature 1": p[0], "Feature 2": p[1], "Correlation": f"{p[3]:+.3f}"}
    for p in pairs[:5]
])
print("Top high-level feature correlations:")
print(top_corr_df.to_string(index=False))


## 9. Synthesis and implications for machine learning modeling

**Question:** What modeling decisions emerge from the complete 11-million-event analysis?

**Expected output:** A concise structured summary of empirical modeling principles, with all measurements based on the complete raw dataset.


In [ ]:
summary_insights = pd.DataFrame([
    {"Aspect": "Class Balance", "Finding": f"{signal_fraction:.1%} signal / {1 - signal_fraction:.1%} background", "Implication": "Standard cross-entropy loss without class re-weighting"},
    {"Aspect": "Data Quality", "Finding": f"{total_nulls:,} missing and {infinite_counts:,} infinite feature values in this analysis sample", "Implication": "No imputation needed; raw tensors ready for scaling"},
    {"Aspect": "Feature Scales", "Finding": "pT and invariant masses have long right tails", "Implication": "StandardScaler in Pipeline required for Logistic Regression"},
    {"Aspect": "Feature Groups", "Finding": "Compare measured feature-group separation before claiming which group is better", "Implication": "Benchmark low_level vs high_level vs all to test representation learning"},
])
print(summary_insights.to_string(index=False))


## 10. Export report-ready EDA evidence

The notebook output is intentionally saved as small, inspectable artifacts rather than relying on transient cell output. These files are the source for the report's EDA tables and figures, while the raw and processed datasets remain local and ignored by git.


In [ ]:
from higgs_event_lab.eda import save_eda_outputs
eda_report = save_eda_outputs(X, y, output_dir=root / "outputs" / "eda", data_source=str(data_path))
print("EDA evidence written to:", root / "outputs" / "eda")
print("Rows:", eda_report["rows"], "Signal fraction:", f'{eda_report["signal_fraction"]:.4f}')
print("Figures:", ", ".join(eda_report["figures"]))
